# Time-Table Optimizer — Step 2 Data Inspection

This notebook inspects the simulated Step 2 timetable scenarios, performs cleaning and validation, and records evidence.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'timetable_scenarios_raw.csv'
INVALID_FILE = PROJECT_ROOT / 'data' / 'raw' / 'timetable_invalid_test_scenarios_raw.csv'

df = pd.read_csv(RAW_FILE)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw file: {RAW_FILE.relative_to(PROJECT_ROOT)}')

Project root: c:\Users\shour\Desktop\Time-Table Optimizer\Time-Table-Optimizer
Raw file: data\raw\timetable_scenarios_raw.csv


In [2]:
print('RAW DATA INSPECTION')
print('-' * 45)
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values per column:')
print(df.isna().sum())
print('\nExact duplicate rows:', int(df.duplicated().sum()))
print('\nScenario categories:')
print(df[['scenario_id', 'case_type']].drop_duplicates()['case_type'].value_counts())
print('\nFirst five rows:')
display(df.head())

RAW DATA INSPECTION
---------------------------------------------
Shape: (50100, 11)

Columns:
['scenario_id', 'case_type', 'course_id', 'cohort_id', 'faculty_id', 'weekly_sessions', 'duration_slots', 'required_room_type', 'room_capacity', 'room_type', 'slot_id']

Missing values per column:
scenario_id            0
case_type              0
course_id              0
cohort_id              0
faculty_id             0
weekly_sessions       50
duration_slots         0
required_room_type     0
room_capacity          0
room_type              0
slot_id                0
dtype: int64

Exact duplicate rows: 100

Scenario categories:
case_type
normal      7000
boundary    2000
stress      1000
Name: count, dtype: int64

First five rows:


,scenario_id,case_type,course_id,cohort_id,faculty_id,weekly_sessions,duration_slots,required_room_type,room_capacity,room_type,slot_id
0,TTO-00001,normal,C01,G1,F1,NaN,2,classroom,60,classroom,S4
1,TTO-00001,normal,C02,G1,F3,NaN,1,lab,30,lab,S1
2,TTO-00001,normal,C03,G1,F1,NaN,1,classroom,60,classroom,S9
3,TTO-00001,normal,C04,G1,F3,NaN,2,classroom,60,classroom,S8
4,TTO-00001,normal,C05,G2,F1,NaN,2,lab,30,lab,S5


## Cleaning step

The raw dataframe is kept unchanged. Cleaning is performed on a copy.

In [3]:
REQUIRED_COLUMNS = [
    'scenario_id', 'case_type', 'course_id', 'cohort_id', 'faculty_id',
    'weekly_sessions', 'duration_slots', 'required_room_type',
    'room_capacity', 'room_type', 'slot_id'
]
NUMERIC_COLUMNS = ['weekly_sessions', 'duration_slots', 'room_capacity']

clean_df = df.copy()
rows_before = len(clean_df)
missing_before = int(clean_df[NUMERIC_COLUMNS].isna().sum().sum())
duplicates_before = int(clean_df.duplicated().sum())

print('Rows before cleaning:', rows_before)
print('Missing numeric cells before cleaning:', missing_before)
print('Duplicate rows before cleaning:', duplicates_before)

Rows before cleaning: 50100
Missing numeric cells before cleaning: 50
Duplicate rows before cleaning: 100


In [4]:
for column in NUMERIC_COLUMNS:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

clean_df['weekly_sessions'] = clean_df['weekly_sessions'].round().astype(int)
clean_df['duration_slots'] = clean_df['duration_slots'].round().astype(int)
clean_df['room_capacity'] = clean_df['room_capacity'].round().astype(int)

clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print('DATA CLEANING SUMMARY')
print('-' * 45)
print('Rows before cleaning:', rows_before)
print('Rows after cleaning:', len(clean_df))
print('Duplicates removed:', rows_before - len(clean_df))
print('Missing values after cleaning:')
print(clean_df[REQUIRED_COLUMNS].isna().sum())
print('\nCleaned preview:')
display(clean_df.head())

DATA CLEANING SUMMARY
---------------------------------------------
Rows before cleaning: 50100
Rows after cleaning: 50000
Duplicates removed: 100
Missing values after cleaning:
scenario_id           0
case_type             0
course_id             0
cohort_id             0
faculty_id            0
weekly_sessions       0
duration_slots        0
required_room_type    0
room_capacity         0
room_type             0
slot_id               0
dtype: int64

Cleaned preview:


,scenario_id,case_type,course_id,cohort_id,faculty_id,weekly_sessions,duration_slots,required_room_type,room_capacity,room_type,slot_id
0,TTO-00001,normal,C01,G1,F1,3,2,classroom,60,classroom,S4
1,TTO-00001,normal,C02,G1,F3,3,1,lab,30,lab,S1
2,TTO-00001,normal,C03,G1,F1,3,1,classroom,60,classroom,S9
3,TTO-00001,normal,C04,G1,F3,3,2,classroom,60,classroom,S8
4,TTO-00001,normal,C05,G2,F1,3,2,lab,30,lab,S5


## Validation contract

In [5]:
VALID_COURSES = {f'C0{i}' for i in range(1, 6)}
VALID_COHORTS = {'G1', 'G2'}
VALID_FACULTY = {'F1', 'F2', 'F3'}
VALID_ROOM_TYPES = {'classroom', 'lab'}
VALID_SLOTS = {f'S{i}' for i in range(1, 11)}
VALID_CASE_TYPES = {'normal', 'boundary', 'stress'}

validation = {}
validation['missing_required_columns'] = len([c for c in REQUIRED_COLUMNS if c not in clean_df.columns])
validation['rows_with_missing_values'] = int(clean_df[REQUIRED_COLUMNS].isna().any(axis=1).sum())
validation['duplicate_rows'] = int(clean_df.duplicated().sum())
validation['invalid_case_types'] = int((~clean_df['case_type'].isin(VALID_CASE_TYPES)).sum())
validation['invalid_course_ids'] = int((~clean_df['course_id'].isin(VALID_COURSES)).sum())
validation['invalid_cohort_ids'] = int((~clean_df['cohort_id'].isin(VALID_COHORTS)).sum())
validation['invalid_faculty_ids'] = int((~clean_df['faculty_id'].isin(VALID_FACULTY)).sum())
validation['invalid_weekly_sessions'] = int((~clean_df['weekly_sessions'].between(1, 4)).sum())
validation['invalid_duration_slots'] = int((~clean_df['duration_slots'].isin([1, 2])).sum())
validation['invalid_required_room_types'] = int((~clean_df['required_room_type'].isin(VALID_ROOM_TYPES)).sum())
validation['invalid_room_types'] = int((~clean_df['room_type'].isin(VALID_ROOM_TYPES)).sum())
validation['invalid_room_capacity'] = int((clean_df['room_capacity'] <= 0).sum())
validation['invalid_slot_ids'] = int((~clean_df['slot_id'].isin(VALID_SLOTS)).sum())
validation['room_capacity_type_mismatch'] = int(((clean_df['room_type'] == 'classroom') & (clean_df['room_capacity'] != 60) | (clean_df['room_type'] == 'lab') & (clean_df['room_capacity'] != 30)).sum())
validation['required_room_type_mismatch'] = int((clean_df['required_room_type'] != clean_df['room_type']).sum())

scenario_sizes = clean_df.groupby('scenario_id').size()
validation['scenarios_not_equal_to_5_rows'] = int((scenario_sizes != 5).sum())
scenario_course_check = clean_df.groupby('scenario_id')['course_id'].apply(lambda s: set(s) == VALID_COURSES)
validation['scenarios_missing_course'] = int((~scenario_course_check).sum())

print('CLEANED DATA VALIDATION')
print('-' * 45)
for name, count in validation.items():
    print(f'{name}: {count}')

if all(value == 0 for value in validation.values()):
    print('\nVALIDATION PASSED')
else:
    print('\nVALIDATION FAILED')

CLEANED DATA VALIDATION
---------------------------------------------
missing_required_columns: 0
rows_with_missing_values: 0
duplicate_rows: 0
invalid_case_types: 0
invalid_course_ids: 0
invalid_cohort_ids: 0
invalid_faculty_ids: 0
invalid_weekly_sessions: 0
invalid_duration_slots: 0
invalid_required_room_types: 0
invalid_room_types: 0
invalid_room_capacity: 0
invalid_slot_ids: 0
room_capacity_type_mismatch: 0
required_room_type_mismatch: 0
scenarios_not_equal_to_5_rows: 0
scenarios_missing_course: 0

VALIDATION PASSED


In [ ]:
scenario_summary = clean_df[['scenario_id', 'case_type']].drop_duplicates()
print('SCENARIO SUMMARY')
print('-' * 45)
print('Distinct scenarios:', scenario_summary['scenario_id'].nunique())
print('\nScenario categories:')
print(scenario_summary['case_type'].value_counts())
print('\nCourses per scenario:')
print(clean_df.groupby('scenario_id').size().value_counts().sort_index())

## Invalid testing scenarios

The invalid file is separate. It is never mixed into the cleaned valid scenario dataset.

In [8]:
invalid_df = pd.read_csv(INVALID_FILE)

def invalid_row_errors(row):
    errors = []

    if row["course_id"] not in VALID_COURSES:
        errors.append("invalid_course_id")

    if row["cohort_id"] not in VALID_COHORTS:
        errors.append("invalid_cohort_id")

    if row["faculty_id"] not in VALID_FACULTY:
        errors.append("invalid_faculty_id")

    if not (1 <= row["weekly_sessions"] <= 4):
        errors.append("invalid_weekly_sessions")

    if row["duration_slots"] not in [1, 2]:
        errors.append("invalid_duration_slots")

    if row["required_room_type"] not in VALID_ROOM_TYPES:
        errors.append("invalid_required_room_type")

    if row["room_type"] not in VALID_ROOM_TYPES:
        errors.append("invalid_room_type")

    if row["room_capacity"] <= 0:
        errors.append("invalid_room_capacity")

    if row["slot_id"] not in VALID_SLOTS:
        errors.append("invalid_slot_id")

    return errors


invalid_df["detected_errors"] = invalid_df.apply(
    invalid_row_errors,
    axis=1
)

# Check whether each scenario contains at least one detected error
scenario_detected = (
    invalid_df.groupby("scenario_id")["detected_errors"]
    .apply(lambda errors: any(bool(error_list) for error_list in errors))
)

print("INVALID SCENARIO TEST")
print("-" * 45)

print(
    "Invalid scenarios detected:",
    int(scenario_detected.sum()),
    "of",
    len(scenario_detected)
)

print("\nExpected invalid reasons:")
print(invalid_df["invalid_reason"].value_counts())

assert scenario_detected.all(), (
    "At least one intentionally invalid scenario was not detected."
)

print("\nINVALID TEST SCENARIOS DETECTED SUCCESSFULLY")

INVALID SCENARIO TEST
---------------------------------------------
Invalid scenarios detected: 100 of 100

Expected invalid reasons:
invalid_reason
invalid_course_id          100
invalid_faculty_id         100
invalid_weekly_sessions    100
invalid_duration_slots     100
invalid_slot_id            100
Name: count, dtype: int64

INVALID TEST SCENARIOS DETECTED SUCCESSFULLY


In [9]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(PROCESSED_DIR / 'cleaned_scenarios.csv', index=False)
clean_df[clean_df['case_type'] == 'normal'].to_csv(PROCESSED_DIR / 'normal_scenarios.csv', index=False)
clean_df[clean_df['case_type'] == 'boundary'].to_csv(PROCESSED_DIR / 'boundary_scenarios.csv', index=False)
clean_df[clean_df['case_type'] == 'stress'].to_csv(PROCESSED_DIR / 'stress_scenarios.csv', index=False)

print('PROCESSED FILES SAVED')
for path in sorted(PROCESSED_DIR.glob('*.csv')):
    print('-', path.relative_to(PROJECT_ROOT))

PROCESSED FILES SAVED
- data\processed\boundary_scenarios.csv
- data\processed\cleaned_scenarios.csv
- data\processed\normal_scenarios.csv
- data\processed\stress_scenarios.csv
